# 01 — Data Pipeline

`00_data_extraction.ipynb`가 만든 `demos_ant.npz`를 train/val split하고 train-only normalization stats를 저장합니다.

이 노트북은 **실행만 담당**합니다. split, normalization, Dataset/DataLoader 구성, phase diagnostic plot 로직은 `src/data_pipeline.py`와 `src/dataset.py`에 있습니다.


## 1. Project paths and imports

In [ ]:
import sys
from pathlib import Path

REPO_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = next((p for p in REPO_ROOT_CANDIDATES if (p / 'src' / 'paths.py').exists()), None)
assert REPO_ROOT is not None, 'repo root with src/paths.py not found; run this notebook from the cloned repository'
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from data_pipeline import (
    DataPipelineConfig,
    plot_phase_advance,
    run_data_pipeline,
)
from paths import ARTIFACT_ROOT, DATA_DIR, FIGURES_DIR, ensure_artifact_dirs

ensure_artifact_dirs()
DATA_PATH = DATA_DIR / 'demos_ant.npz'
NORM_PATH = DATA_DIR / 'norm_stats.npz'
assert DATA_PATH.exists(), f'파일 없음: {DATA_PATH}'
print(f'✓ src 경로 등록: {SRC_DIR}')
print(f'✓ artifact root: {ARTIFACT_ROOT}')
print(f'✓ 데이터 발견: {DATA_PATH}')

## 2. Run deterministic data pipeline

In [ ]:
config = DataPipelineConfig(
    obs_horizon=2,
    pred_horizon=16,
    action_horizon=8,
    val_ratio=0.15,
    batch_size=256,
    num_workers=2,
    seed=42,
)

pipeline = run_data_pipeline(DATA_PATH, NORM_PATH, config)
train_dataset = pipeline['train_dataset']
val_dataset = pipeline['val_dataset']
train_loader = pipeline['train_loader']
val_loader = pipeline['val_loader']

## 3. Phase diagnostic


## 4. Diagnostic figures

In [ ]:
phase_advance = plot_phase_advance(
    train_dataset,
    FIGURES_DIR,
    f_mean=pipeline['project_data']['freq_window_mean'],
    seed=config.seed,
)
print('\n=== 01 Data Pipeline 완료 ===')
print(f'Train chunks: {len(train_dataset)}')
print(f'Val chunks:   {len(val_dataset)}')
print(f'Steps/epoch:  {len(train_loader)}')
print(f'Norm stats:   {NORM_PATH}')
print(f'Phase plot:   {phase_advance["path"]}')
